# 🚀 Полный практический тренажер по NumPy и Pandas для ML-инженера

В этом ноутбуке собраны главные концепции, функции и трюки **NumPy** и **Pandas**, которые реально используются в пайплайнах предобработки данных, Feature Engineering и на технических собеседованиях.

--- 
## 📚 Глобальная шпаргалка (Все нужные функции и методы)

### 1. NumPy: Векторизация, матрицы и линейная алгебра
* **Создание и форма:** `np.array()`, `np.zeros()`, `np.ones()`, `np.full()`, `np.eye()`, `np.arange()`, `np.linspace()`, `.reshape()`, `.ravel()`, `.flatten()`, `.T` (транспонирование), `np.expand_dims()`, `np.squeeze()`.
* **Объединение и разбиение:** `np.concatenate()`, `np.vstack()`, `np.hstack()`, `np.column_stack()`, `np.split()`.
* **Условия и фильтрация:** `np.where(cond, x, y)`, `np.select()`, `np.clip(a, min, max)`, `np.isnan()`, `np.isinf()`, `np.nan_to_num()`, булевы маски `arr[arr > 0]`.
* **Агрегации (с поддержкой `axis=0/1` и `keepdims=True`):** `np.sum()`, `np.mean()`, `np.std()`, `np.var()`, `np.min()`, `np.max()`, `np.median()`, `np.percentile()`, `np.quantile()`, `np.cumsum()`, `np.cumprod()`, а также `np.nanmean()`, `np.nansum()`.
* **Поиск и сортировка:** `np.argmax()`, `np.argmin()`, `np.argsort()`, `np.sort()`, `np.searchsorted()`, `np.unique(return_counts=True)`.
* **Линейная алгебра и математика:** `@` / `np.dot()` / `np.matmul()`, `np.linalg.norm()`, `np.linalg.inv()`, `np.linalg.eig()`, `np.einsum()`, `np.exp()`, `np.log()`, `np.log1p()`.
* **Продвинутая индексация:** Fancy Indexing `arr[rows, cols]`, `np.take_along_axis()`, `np.put_along_axis()`, Broadcasting (правила совместимости размерностей справа налево).

### 2. Pandas: Табличные данные и Feature Engineering
* **Чтение и инспекция:** `pd.read_csv()`, `pd.read_parquet()`, `.head()`, `.tail()`, `.info()`, `.describe()`, `.dtypes`, `.shape`, `.value_counts(normalize=True)`, `.nunique()`, `.unique()`.
* **Индексация и фильтрация:** `.loc[row_labels, col_labels]` (по меткам/булевым маскам), `.iloc[row_idx, col_idx]` (по целочисленным позициям), `.query()`, `.isin()`, `.between()`, `.set_index()`, `.reset_index()`.
* **Работа с пропусками и дубликатами:** `.isna()`, `.notna()`, `.fillna()`, `.dropna()`, `.interpolate()`, `.ffill()`, `.bfill()`, `.duplicated()`, `.drop_duplicates()`.
* **Преобразование колонок и типов:** `.astype()`, `pd.to_numeric()`, `pd.to_datetime()`, `.rename()`, `.drop()`, `.assign()`, `.map()`, `.apply()`, `np.where()`, `.clip()`, `pd.cut()`, `pd.qcut()`, `pd.get_dummies()`.
* **Строки (`.str`) и Даты (`.dt`):**
  * `.str.lower()`, `.str.strip()`, `.str.contains()`, `.str.extract()`, `.str.replace()`, `.str.split()`, `.str.len()`.
  * `.dt.year`, `.dt.month`, `.dt.day`, `.dt.dayofweek`, `.dt.hour`, `.dt.total_seconds()`, `.dt.floor()`.
* **Группировка и агрегация (`groupby`):**
  * `df.groupby(cols).agg({'col1': ['mean', 'max'], 'col2': 'count'})`
  * `df.groupby(cols)['col'].transform('mean')` — возвращает Series длины исходного DataFrame!
  * `df.groupby(cols)['col'].shift(1)` / `.diff(1)` / `.cumsum()`.
* **Оконные функции:** `.rolling(window, min_periods=1).mean()`, `.expanding().sum()`, `.ewm(span=...).mean()`.
* **Слияние и изменение структуры:** `pd.concat()`, `pd.merge(how='left'/'inner'/'outer')`, `df.pivot_table()`, `df.melt()`.

---
## 🛠 Шаг 0. Генерация датасета для тренировок
Запусти ячейку ниже, чтобы импортировать библиотеки и сгенерировать синтетический датасет пользовательских сессий `df_sessions` и справочник пользователей `df_users`.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

# Справочник пользователей
df_users = pd.DataFrame({
    "user_id": [101, 102, 103, 104, 105, 106],
    "segment": ["VIP", "Regular", "Regular", "New", "VIP", "Regular"],
    "city": ["Moscow", "SPb", "Moscow", "Kazan", "SPb", "Moscow"]
})

# Таблица сессий и покупок
df_sessions = pd.DataFrame({
    "session_id": range(1, 13),
    "user_id": [101, 102, 101, 103, 104, 102, 105, 101, 106, 103, 105, 102],
    "timestamp": [
        "2026-03-01 10:15:00", "2026-03-01 12:30:00", "2026-03-02 18:45:00",
        "2026-03-02 23:10:00", "2026-03-03 09:05:00", "2026-03-04 14:20:00",
        "2026-03-05 19:50:00", "2026-03-06 08:00:00", "2026-03-06 16:40:00",
        "2026-03-07 21:15:00", "2026-03-08 11:10:00", "2026-03-08 15:00:00"
    ],
    "device": [
        "  iOS_v16.2 ", "Android_v13.0", "iOS_v16.2", "Desktop_Win11",
        "android_v12.1  ", "Android_v13.0", "macOS_v14.0", "iOS_v16.3",
        "Desktop_Win10", "Desktop_Win11", "  macOS_v14.0", "Android_v13.0"
    ],
    "duration_sec": [120.0, 45.0, 310.0, np.nan, 15.0, 180.0, 600.0, 95.0, 40.0, 210.0, np.nan, 150.0],
    "pages_viewed": [5, 2, 12, 1, 1, 6, 20, 4, 2, 8, 15, 5],
    "amount": [1500.0, 0.0, 3200.0, 0.0, 0.0, 850.0, 12500.0, 500.0, 0.0, 1900.0, 4300.0, 1200.0],
    "target_converted": [1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1]
})

print("✅ Датасеты df_users и df_sessions успешно созданы!")
df_sessions.head()

✅ Датасеты df_users и df_sessions успешно созданы!


,session_id,user_id,timestamp,device,duration_sec,pages_viewed,amount,target_converted
0,1,101,2026-03-01 10:15:00,iOS_v16.2,120.0,5,1500.0,1
1,2,102,2026-03-01 12:30:00,Android_v13.0,45.0,2,0.0,0
2,3,101,2026-03-02 18:45:00,iOS_v16.2,310.0,12,3200.0,1
3,4,103,2026-03-02 23:10:00,Desktop_Win11,NaN,1,0.0,0
4,5,104,2026-03-03 09:05:00,android_v12.1,15.0,1,0.0,0


---
# ЧАСТЬ 1. NUMPY (Векторизация и матричные операции)

## Задача 1. Стандартизация (Z-score) и борьба с выбросами (Clipping)
**Концепции:** `np.mean`, `np.std`, `keepdims=True`, `np.where`, `np.clip`, Broadcasting.

В ML перед подачей признаков в линейную модель или нейросеть их стандартизуют по колонкам (признакам): 
$$X_{\text{norm}} = \frac{X - \mu}{\sigma}$$
Если в какой-то колонке все значения одинаковые ($\sigma = 0$), деление даст `NaN`/`Inf` — такие колонки нужно занулять.

**Задание:** Напиши функцию `preprocess_features(X, clip_val=3.0)`, которая:
1. Принимает 2D-массив `X` размера `(N, D)`.
2. Вычисляет среднее и стандартное отклонение по каждой колонке (`axis=0`, `keepdims=True`).
3. Стандартизует матрицу (если `std == 0`, вся колонка должна стать нулями).
4. Обрезает экстремальные выбросы в итоговой матрице в диапазон `[-clip_val, clip_val]` с помощью `np.clip`.

In [6]:
def preprocess_features(X: np.ndarray, clip_val: float = 3.0) -> np.ndarray:
    # ТВОЙ КОД ЗДЕСЬ
    means,stds = np.mean(X,axis=0,keepdims=True), np.std(X,axis=0,keepdims=True)
    stds = np.where(stds == 0, 1.0, stds)
    X = (X - means) / stds
    X = np.where(stds == 0, 0.0, X)
    X = np.clip(X,-clip_val,clip_val)
    return X
    # pass


# === ТЕСТЫ ===
X_test = np.array([
    [10.0, 5.0, 1.0],
    [20.0, 5.0, 2.0],
    [30.0, 5.0, 100.0]
])
res = preprocess_features(X_test, clip_val=1.2)
assert res.shape == (3, 3), "Неверная форма массива!"
assert np.allclose(res[:, 1], 0.0), "Константная колонка должна стать нулевой!"
assert np.allclose(res[:, 0], [-1.2, 0.0, 1.2]), f"Ошибка в 1-й колонке: {res[:, 0]}"
assert np.max(np.abs(res)) <= 1.2, "Clipping не сработал!"
print("✅ Задача 1 решена верно!")

✅ Задача 1 решена верно!


---
## Задача 2. Численно стабильный Softmax и Cross-Entropy Loss (Fancy Indexing)
**Концепции:** `np.max`, `np.exp`, `np.sum`, `np.log`, Fancy Indexing `probs[np.arange(N), y_true]`.

При вычислении Softmax напрямую ($e^{z_i} / \sum e^{z_j}$) большие числа вызывают переполнение (`inf`). Чтобы этого избежать, из каждой строки вычитают максимум этой строки.

**Задание:** Реализуй две функции **без циклов `for`**:
1. `stable_softmax(logits)` — принимает 2D-массив `logits` размера `(N, C)` и возвращает матрицу вероятностей `(N, C)`.
2. `cross_entropy_loss(probs, y_true)` — принимает матрицу вероятностей `(N, C)` и 1D-массив истинных индексов классов `y_true` длины `N`. Возвращает среднее значение $-rac{1}{N} \sum_{i=1}^N \ln(p_{i, y_i})$.

In [10]:
def stable_softmax(logits: np.ndarray) -> np.ndarray:
    # ТВОЙ КОД ЗДЕСЬ
    logits = logits - np.max(logits,axis=1,keepdims=True)
    logits = np.exp(logits) / np.sum(np.exp(logits),axis=1,keepdims=True)
    return logits
    # pass

def cross_entropy_loss(probs: np.ndarray, y_true: np.ndarray) -> float:
    # ТВОЙ КОД ЗДЕСЬ (используй индексацию probs[np.arange(len(y_true)), y_true])
    return -np.mean(np.log(probs[np.arange(len(y_true)),y_true]))
    


# === ТЕСТЫ ===
logits = np.array([[1000.0, 1000.0, 1000.0], [1.0, 2.0, 3.0]])
probs = stable_softmax(logits)
assert not np.isnan(probs).any(), "Переполнение! Забыл вычесть максимум по строке."
assert np.allclose(probs[0], [1/3, 1/3, 1/3]), "Неверные вероятности для первой строки!"
assert np.allclose(np.sum(probs, axis=1), [1.0, 1.0]), "Сумма вероятностей должна быть равна 1!"

y_true = np.array([0, 2])
loss = cross_entropy_loss(probs, y_true)
expected_loss = float(-np.mean([np.log(1/3), np.log(probs[1, 2])]))
assert np.isclose(loss, expected_loss), f"Ожидался loss {expected_loss}, получено {loss}"
print("✅ Задача 2 решена верно!")

✅ Задача 2 решена верно!


---
## Задача 3. Матрица попарных расстояний и Косинусное сходство
**Концепции:** `@` (матричное умножение), `np.linalg.norm`, алгебраическое раскрытие квадрата расстояния.

В рекомендательных системах и компьютерном зрении постоянно нужно считать сходство между эмбеддингами. 

**Задание:** Напиши функцию `cosine_similarity_matrix(A, B)`, которая принимает две матрицы эмбеддингов `A` размера `(N, D)` и `B` размера `(M, D)` и возвращает матрицу косинусных сходств размера `(N, M)` без единого цикла:
$$\text{CosSim}(a, b) = \frac{a \cdot b}{\|a\|_2 \|b\|_2}$$

In [15]:
def cosine_similarity_matrix(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    # ТВОЙ КОД ЗДЕСЬ
    return (A @ B.T) / (np.linalg.norm(A,axis=1,keepdims=True) * np.linalg.norm(B,axis=1,keepdims=True).T)
    # pass


# === ТЕСТЫ ===
A = np.array([[1.0, 0.0], [0.0, 2.0], [1.0, 1.0]])
B = np.array([[2.0, 0.0], [0.0, 1.0]])
sim = cosine_similarity_matrix(A, B)
expected = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1 / np.sqrt(2), 1 / np.sqrt(2)]
])
assert sim.shape == (3, 2), f"Неверная размерность: {sim.shape}"
assert np.allclose(sim, expected), f"Неверные значения! Получено:\n{sim}"
print("✅ Задача 3 решена верно!")

✅ Задача 3 решена верно!


---
## Задача 4. Топ-K предсказаний и быстрая квантизация
**Концепции:** `np.argsort`, срезы с шагом `[..., ::-1]`, `np.take_along_axis`, `np.unique`.

**Задание:** Напиши функцию `get_top_k_predictions(probs, k)`, которая принимает матрицу вероятностей размера `(N, C)` и возвращает кортеж из двух массивов размера `(N, k)`:
1. `top_k_indices` — индексы `k` самых вероятных классов для каждого объекта, отсортированные **по убыванию** вероятности.
2. `top_k_probs` — сами значения вероятностей, соответствующие этим индексам.

In [20]:
def get_top_k_predictions(probs: np.ndarray, k: int) -> tuple[np.ndarray, np.ndarray]:
    # ТВОЙ КОД ЗДЕСЬ
    return (np.argsort(probs,axis=1)[:,::-1][:,:k],probs[np.arange(len(probs))[:,None],np.argsort(probs,axis=1)[:,::-1][:,:k]])
    # pass


# === ТЕСТЫ ===
probs_test = np.array([
    [0.1, 0.5, 0.3, 0.1],
    [0.4, 0.1, 0.05, 0.45]
])
idx, val = get_top_k_predictions(probs_test, k=2)
assert np.array_equal(idx, np.array([[1, 2], [3, 0]])), f"Неверные индексы: {idx}"
assert np.allclose(val, np.array([[0.5, 0.3], [0.45, 0.4]])), f"Неверные вероятности: {val}"
print("✅ Задача 4 решена верно!")

✅ Задача 4 решена верно!


---
## Задача 5. Обработка NaN в матрицах и добавление Bias-колонки
**Концепции:** `np.isnan`, `np.nanmean`, `np.where`, `np.ones`, `np.hstack` / `np.column_stack`.

**Задание:** Напиши функцию `impute_and_add_bias(X)`, которая:
1. Заменяет все `np.nan` в каждой колонке 2D-матрицы `X` на среднее значение этой колонки (используй `np.nanmean(X, axis=0)`).
2. Добавляет **слева** (нулевой колонкой) столбец из единиц `1.0` для свободного члена линейной регрессии (bias/intercept).

In [25]:
def impute_and_add_bias(X: np.ndarray) -> np.ndarray:
    # ТВОЙ КОД ЗДЕСЬ
    X = np.where(np.isnan(X), np.nanmean(X,axis=0),X)
    X = np.hstack([np.ones((len(X),1)),X])
    return X
    # pass


# === ТЕСТЫ ===
X_nan = np.array([
    [1.0, np.nan],
    [3.0, 10.0],
    [np.nan, 20.0]
])
res = impute_and_add_bias(X_nan)
expected = np.array([
    [1.0, 1.0, 15.0],
    [1.0, 3.0, 10.0],
    [1.0, 2.0, 20.0]
])
assert np.allclose(res, expected), f"Ошибка! Ожидалось:\n{expected}\nПолучено:\n{res}"
print("✅ Задача 5 решена верно!")

✅ Задача 5 решена верно!


---
# ЧАСТЬ 2. PANDAS (Анализ, очистка и Feature Engineering)

## Задача 6. Продвинутая фильтрация: `.loc`, `.isin`, `.between`
**Концепции:** Булева индексация, `.isin()`, `.between()`, `.loc[]`, `.reset_index(drop=True)`.

**Задание:** Напиши функцию `filter_active_sessions(df, valid_users)`, которая:
1. Принимает датафрейм `df` (структуры `df_sessions`) и список ID пользователей `valid_users`.
2. Оставляет только строки, где:
   * `user_id` входит в список `valid_users` (`.isin()`),
   * `pages_viewed` находится в диапазоне от 4 до 15 включительно (`.between(4, 15)`),
   * `amount` строго больше 0.
3. Возвращает отфильтрованный датафрейм, содержащий только 3 колонки: `['session_id', 'user_id', 'amount']`, со сброшенным индексом (`.reset_index(drop=True)`).

In [27]:
def filter_active_sessions(df: pd.DataFrame, valid_users: list[int]) -> pd.DataFrame:
    # ТВОЙ КОД ЗДЕСЬ
    df = df[df['user_id'].isin(valid_users)]
    df = df[(4 <= df['pages_viewed']) & (df['pages_viewed'] <= 15) & (df['amount'] > 0)]
    return df[['session_id','user_id','amount']].reset_index(drop=True)

    # pass


# === ТЕСТЫ ===
sub_df = filter_active_sessions(df_sessions, valid_users=[101, 102, 105])
assert list(sub_df.columns) == ["session_id", "user_id", "amount"], "Неверный набор колонок!"
assert list(sub_df["session_id"]) == [1, 3, 6, 8, 11, 12], f"Неверные сессии: {list(sub_df['session_id'])}"
assert list(sub_df.index) == [0, 1, 2, 3, 4, 5], "Забыл сделать reset_index(drop=True)!"
print("✅ Задача 6 решена верно!")

✅ Задача 6 решена верно!


---
## Задача 7. Очистка строк и парсинг признаков (`.str` accessor)
**Концепции:** `.str.strip()`, `.str.lower()`, `.str.split()`, `.str.extract()`, `.astype()`.

Посмотри на колонку `device` в `df_sessions`: там есть лишние пробелы, разный регистр и склеенная через `_v` версия ОС (`"  iOS_v16.2 "`, `"android_v12.1  "`).

**Задание:** Напиши функцию `parse_device_column(df)`, которая не меняет исходный датафрейм (сделай `df = df.copy()`), а добавляет две новые колонки:
1. `os_name` — название ОС в нижнем регистре без пробелов по краям (все, что идет до `_v` или `_Win`).
2. `is_mobile` — булева колонка (`True`, если `os_name` равен `'ios'` или `'android'`, иначе `False`).

In [31]:
def parse_device_column(df: pd.DataFrame) -> pd.DataFrame:
    # ТВОЙ КОД ЗДЕСЬ
    df = df.copy()
    df['os_name'] = df['device'].str.lower().str.strip().str.split('_').str[0]
    df['is_mobile'] = np.where((df['os_name'] == 'ios') | (df['os_name'] == 'android'),True,False)
    return df
    # pass


# === ТЕСТЫ ===
parsed_df = parse_device_column(df_sessions)
assert list(parsed_df["os_name"].unique()) == ["ios", "android", "desktop", "macos"], f"Ошибка в os_name: {parsed_df['os_name'].unique()}"
assert parsed_df["is_mobile"].sum() == 7, f"Ожидалось 7 мобильных сессий, получено {parsed_df['is_mobile'].sum()}"
print("✅ Задача 7 решена верно!")

✅ Задача 7 решена верно!


---
## Задача 8. Работа со временем и датами (`.dt` accessor)
**Концепции:** `pd.to_datetime()`, `.dt.hour`, `.dt.dayofweek`, `.groupby().diff()`, `.dt.total_seconds()`.

**Задание:** Напиши функцию `extract_time_features(df)`, которая на копии датафрейма:
1. Переводит колонку `timestamp` в формат `datetime`.
2. Сортирует строки по `['user_id', 'timestamp']` по возрастанию.
3. Создает колонку `hour` — час начала сессии.
4. Создает колонку `is_weekend` — `1`, если день недели суббота (5) или воскресенье (6), иначе `0` (`.astype(int)`).
5. Создает колонку `hours_since_prev_session` — сколько часов прошло с предыдущей сессии **этого же пользователя** (используй `df.groupby('user_id')['timestamp'].diff().dt.total_seconds() / 3600`). Первую сессию каждого юзера заполни значением `-1.0`.

In [33]:
def extract_time_features(df: pd.DataFrame) -> pd.DataFrame:
    # ТВОЙ КОД ЗДЕСЬ
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values(["user_id", "timestamp"])
    df["hour"] = df["timestamp"].dt.hour
    df["is_weekend"] = (df["timestamp"].dt.dayofweek >= 5).astype(int)
    diff_hours = df.groupby("user_id")["timestamp"].diff().dt.total_seconds() / 3600.0
    df["hours_since_prev_session"] = diff_hours.fillna(-1.0)
    return df
    # pass


# === ТЕСТЫ ===
time_df = extract_time_features(df_sessions)
u101 = time_df[time_df["user_id"] == 101]
assert list(u101["hour"]) == [10, 18, 8], f"Неверные часы: {list(u101['hour'])}"
assert list(u101["is_weekend"]) == [1, 0, 0], f"Ошибка в is_weekend: {list(u101['is_weekend'])}"
assert np.allclose(u101["hours_since_prev_session"], [-1.0, 32.5, 85.25]), f"Ошибка в разнице часов: {list(u101['hours_since_prev_session'])}"
print("✅ Задача 8 решена верно!")

✅ Задача 8 решена верно!


---
## Задача 9. Групповая импутация пропусков (`groupby` + `transform`)
**Концепции:** `df.groupby().transform()`, `.fillna()`.

В колонке `duration_sec` есть `NaN`. Заполнять их общим средним по всей таблице грубо — лучше заполнить **медианой длительности сессий конкретного пользователя** (или конкретной группы).

**Задание:** Напиши функцию `impute_duration_by_user(df)`, которая возвращает копию датафрейма, где пропуски в `duration_sec` заполнены медианой `duration_sec` по соответствующему `user_id`. Если у пользователя вообще все значения `NaN`, заполни их общей медианой по всему датафрейму.

In [ ]:
def impute_duration_by_user(df: pd.DataFrame) -> pd.DataFrame:
    # ТВОЙ КОД ЗДЕСЬ
    pass


# === ТЕСТЫ ===
imp_df = impute_duration_by_user(df_sessions)
assert imp_df["duration_sec"].isna().sum() == 0, "Остались пропуски!"
# Для user_id=103 единственное известное значение 210.0, для user_id=105 -> 600.0
assert imp_df.loc[imp_df["session_id"] == 4, "duration_sec"].values[0] == 210.0, "Неверная импутация для сессии 4!"
assert imp_df.loc[imp_df["session_id"] == 11, "duration_sec"].values[0] == 600.0, "Неверная импутация для сессии 11!"
print("✅ Задача 9 решена верно!")

<details>
<summary>💡 <b>Показать подсказку и решение (Задача 9)</b></summary>

```python
def impute_duration_by_user(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    user_medians = df.groupby("user_id")["duration_sec"].transform("median")
    global_median = df["duration_sec"].median()
    df["duration_sec"] = df["duration_sec"].fillna(user_medians).fillna(global_median)
    return df
```
</details>

---
## Задача 10. Сборка витрины признаков (`groupby` + Named Aggregation)
**Концепции:** `df.groupby().agg(new_col=('col', 'func'))`, `.reset_index()`.

Классическая задача аналитика и ML-инженера: свернуть лог событий в таблицу профилей пользователей (User-level features).

**Задание:** Напиши функцию `build_user_profile(df)`, которая группирует `df` по `user_id` (с `.reset_index()`) и считает 4 колонки:
1. `total_sessions` — количество сессий пользователя (`count` по `session_id`).
2. `total_spent` — сумма покупок (`sum` по `amount`).
3. `max_pages` — максимум просмотренных страниц за одну сессию (`max` по `pages_viewed`).
4. `conversion_rate` — доля сессий с покупкой (`mean` по `target_converted`), округленная до 2 знаков.

In [ ]:
def build_user_profile(df: pd.DataFrame) -> pd.DataFrame:
    # ТВОЙ КОД ЗДЕСЬ
    pass


# === ТЕСТЫ ===
profiles = build_user_profile(df_sessions)
expected_cols = ["user_id", "total_sessions", "total_spent", "max_pages", "conversion_rate"]
assert list(profiles.columns) == expected_cols, f"Неверные колонки: {list(profiles.columns)}"
u102 = profiles[profiles["user_id"] == 102].iloc[0]
assert u102["total_sessions"] == 3 and u102["total_spent"] == 2050.0 and u102["conversion_rate"] == 0.67, f"Ошибка для user 102: {u102.to_dict()}"
print("✅ Задача 10 решена верно!")

<details>
<summary>💡 <b>Показать подсказку и решение (Задача 10)</b></summary>

```python
def build_user_profile(df: pd.DataFrame) -> pd.DataFrame:
    res = df.groupby("user_id", as_index=False).agg(
        total_sessions=("session_id", "count"),
        total_spent=("amount", "sum"),
        max_pages=("pages_viewed", "max"),
        conversion_rate=("target_converted", "mean")
    )
    res["conversion_rate"] = res["conversion_rate"].round(2)
    return res
```
</details>

---
## Задача 11. Слияние таблиц (`pd.merge`) и Сводные таблицы (`pivot_table`)
**Концепции:** `pd.merge()`, `df.pivot_table()`.

**Задание:** Напиши функцию `city_segment_revenue(df_sess, df_usr)`, которая:
1. Объединяет `df_sess` и `df_usr` по `user_id` (`how='left'`).
2. Строит сводную таблицу (`pivot_table`), где:
   * Индексы (`index`) — города (`city`),
   * Колонки (`columns`) — сегменты (`segment`),
   * Значения (`values`) — сумма покупок (`amount`, `aggfunc='sum'`),
   * Пустые ячейки заполнены нулями (`fill_value=0.0`).

In [ ]:
def city_segment_revenue(df_sess: pd.DataFrame, df_usr: pd.DataFrame) -> pd.DataFrame:
    # ТВОЙ КОД ЗДЕСЬ
    pass


# === ТЕСТЫ ===
piv = city_segment_revenue(df_sessions, df_users)
assert piv.loc["Moscow", "VIP"] == 5200.0, f"Ожидалось 5200.0 для Moscow VIP, получено {piv.loc['Moscow', 'VIP']}"
assert piv.loc["SPb", "VIP"] == 16800.0, "Ошибка для SPb VIP!"
assert piv.loc["Kazan", "VIP"] == 0.0, "Пропуски в pivot_table должны быть заполнены 0.0!"
print("✅ Задача 11 решена верно!")

<details>
<summary>💡 <b>Показать подсказку и решение (Задача 11)</b></summary>

```python
def city_segment_revenue(df_sess: pd.DataFrame, df_usr: pd.DataFrame) -> pd.DataFrame:
    merged = pd.merge(df_sess, df_usr, on="user_id", how="left")
    return merged.pivot_table(
        index="city",
        columns="segment",
        values="amount",
        aggfunc="sum",
        fill_value=0.0
    )
```
</details>

---
## Задача 12. Оконные функции и лаги без утечки из будущего (Data Leakage Prevention)
**Концепции:** `.groupby()`, `.shift(1)`, `.cumsum()`, `.rolling()`.

При создании исторических признаков для конкретной сессии **нельзя** включать в расчет саму текущую сессию, иначе произойдет Target/Data Leakage!

**Задание:** Напиши функцию `add_historical_features(df)`, которая:
1. Сортирует копию `df` по `['user_id', 'session_id']`.
2. Добавляет колонку `prev_amount` — сумму покупки в **предыдущей** сессии этого же пользователя (если сессия первая — `0.0`).
3. Добавляет колонку `cum_amount_before` — накопленную сумму покупок пользователя **строго ДО текущей сессии** (подсказка: посчитай `cumsum()` по группе и вычти текущий `amount`, либо сначала сделай `shift(1, fill_value=0)` и затем `cumsum()`).

In [ ]:
def add_historical_features(df: pd.DataFrame) -> pd.DataFrame:
    # ТВОЙ КОД ЗДЕСЬ
    pass


# === ТЕСТЫ ===
hist_df = add_historical_features(df_sessions)
u101_hist = hist_df[hist_df["user_id"] == 101]
assert list(u101_hist["prev_amount"]) == [0.0, 1500.0, 3200.0], f"Ошибка в prev_amount: {list(u101_hist['prev_amount'])}"
assert list(u101_hist["cum_amount_before"]) == [0.0, 1500.0, 4700.0], f"Ошибка в cum_amount_before: {list(u101_hist['cum_amount_before'])}"
print("✅ Задача 12 решена верно!")

<details>
<summary>💡 <b>Показать подсказку и решение (Задача 12)</b></summary>

```python
def add_historical_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().sort_values(["user_id", "session_id"])
    df["prev_amount"] = df.groupby("user_id")["amount"].shift(1, fill_value=0.0)
    df["cum_amount_before"] = df.groupby("user_id")["prev_amount"].cumsum()
    return df
```
</details>

---
## Задача 13. Сглаженный Target Encoding категориальных признаков
**Концепции:** `.groupby()`, `.map()`, формула сглаживания (Bayesian Mean).

Target Encoding заменяет каждую категорию средним значением таргета в ней. Но если категория встретилась всего 1 раз, мы переобучимся! Поэтому используют сглаживание:
$$\text{TE}(c) = \frac{n_c \cdot \bar{y}_c + m \cdot \bar{y}_{\text{global}}}{n_c + m}$$
где $n_c$ — число объектов категории $c$, $\bar{y}_c$ — средний таргет в категории $c$, $\bar{y}_{\text{global}}$ — глобальный средний таргет по всей выборке, $m$ — вес сглаживания (например, $m=10$).

**Задание:** Напиши функцию `smoothed_target_encoding(df, cat_col, target_col, m=5.0)`, которая возвращает `pd.Series` со сглаженными значениями для каждой строки `df`.

In [ ]:
def smoothed_target_encoding(df: pd.DataFrame, cat_col: str, target_col: str, m: float = 5.0) -> pd.Series:
    # ТВОЙ КОД ЗДЕСЬ
    pass


# === ТЕСТЫ ===
te_series = smoothed_target_encoding(df_sessions, "user_id", "target_converted", m=2.0)
# Глобальное среднее target_converted = 8 / 12 = 2/3.
# У user_id=101: 3 сессии, сумма таргета = 3. TE = (3 + 2 * (2/3)) / (3 + 2) = 4.3333 / 5 = 0.866667
assert np.isclose(te_series.iloc[0], (3 + 2 * (8 / 12)) / 5), f"Неверное значение TE: {te_series.iloc[0]}"
print("✅ Задача 13 решена верно!")

<details>
<summary>💡 <b>Показать подсказку и решение (Задача 13)</b></summary>

```python
def smoothed_target_encoding(df: pd.DataFrame, cat_col: str, target_col: str, m: float = 5.0) -> pd.Series:
    global_mean = df[target_col].mean()
    stats = df.groupby(cat_col)[target_col].agg(["count", "mean"])
    smooth = (stats["count"] * stats["mean"] + m * global_mean) / (stats["count"] + m)
    return df[cat_col].map(smooth)
```
</details>

---
## Задача 14. Биннинг (Дискретизация) и One-Hot Encoding (`pd.cut`, `pd.get_dummies`)
**Концепции:** `pd.cut()`, `pd.qcut()`, `pd.get_dummies()`.

**Задание:** Напиши функцию `bin_and_encode_pages(df)`, которая:
1. Разбивает колонку `pages_viewed` на 3 корзины с помощью `pd.cut` с границами `bins=[0, 3, 10, np.inf]` и метками `labels=['low', 'medium', 'high']`.
2. Применяет `pd.get_dummies(..., prefix='activity', dtype=int)` к полученным корзинам и возвращает датафрейм из 3 бинарных колонок (`activity_low`, `activity_medium`, `activity_high`).

In [ ]:
def bin_and_encode_pages(df: pd.DataFrame) -> pd.DataFrame:
    # ТВОЙ КОД ЗДЕСЬ
    pass


# === ТЕСТЫ ===
ohe_df = bin_and_encode_pages(df_sessions)
assert list(ohe_df.columns) == ["activity_low", "activity_medium", "activity_high"], f"Неверные колонки: {list(ohe_df.columns)}"
assert list(ohe_df.sum()) == [4, 5, 3], f"Неверное распределение по корзинам: {list(ohe_df.sum())}"
print("🎉 ПОЗДРАВЛЯЮ! Все задачи тренажера по NumPy и Pandas успешно решены!")

<details>
<summary>💡 <b>Показать подсказку и решение (Задача 14)</b></summary>

```python
def bin_and_encode_pages(df: pd.DataFrame) -> pd.DataFrame:
    bins = pd.cut(df["pages_viewed"], bins=[0, 3, 10, np.inf], labels=["low", "medium", "high"])
    return pd.get_dummies(bins, prefix="activity", dtype=int)
```
</details>